In [1]:
from dotenv import load_dotenv
import os 
load_dotenv()

True

In [6]:
from typing import TypedDict


class UserContext(TypedDict):
        user_id :str
        plan : str
    

# Useful Things `runtime` Can Give Your Tools

| Runtime Property | What It's Useful For |
|---|---|
| `runtime.context` | App-provided information, such as user ID, subscription plan, permissions, and configuration. |
| `runtime.state` | Current conversation and custom agent state, such as task progress or selected items. |
| `runtime.store` | Accessing long-term data saved across conversations, such as user preferences. |
| `runtime.config` | Run configuration, including the thread ID and other configurable settings. |
| `runtime.tool_call_id` | Identifying the current tool call, especially when returning a `Command` update. |
| `runtime.stream_writer` | Sending progress updates while a tool is running, when streaming is configured. |

In [ ]:
from pprint import pprint

from langchain.agents import create_agent
from langchain.tools import ToolRuntime
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()
# context can be used in tool functions throught Tool runTIME
@tool
def get_user_plan(runtime: ToolRuntime[UserContext]) -> str:
    """Get the plan for the current user."""
    return f"User {runtime.context['user_id']} has the {runtime.context['plan']} plan."


model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="medium",
)


agent = create_agent(
    model=model,
    tools=[get_user_plan],
    context_schema=UserContext,
    system_prompt="You are a helpful assistant. Use get_user_plan to answer questions about the current user's plan.",
)

res =  agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what's user 123 plan?",
            }
        ]
    },
    context={"user_id": "123", "plan": "free"},
)

print(res["messages"][-1].content)

User 123 has the free plan.
